# Publisher economic exposure

This notebook reproduces Table 9 and the sponsored-search-ad counts in §4.4 from raw page analyses and the final paper-defined sponsored-ad indexes.

In [1]:
from pathlib import Path
import json, re
from collections import Counter, defaultdict
from urllib.parse import urlparse

import pandas as pd
from IPython.display import HTML, Markdown, display

ROOT = Path.cwd()
if not (ROOT / "raw").is_dir():
    ROOT = ROOT.parent
RAW_AIOS = ROOT / "raw" / "aio_results"

CATEGORY_NAMES = {
    "autos_vehicles": "Autos & Vehicles", "beauty_fashion": "Beauty & Fashion",
    "business_finance": "Business & Finance", "climate": "Climate",
    "entertainment": "Entertainment", "food_drink": "Food & Drink",
    "games": "Games", "health": "Health", "hobbies_leisure": "Hobbies & Leisure",
    "jobs_education": "Jobs & Education", "law_gov": "Law & Government",
    "other": "Other", "pets_animals": "Pets & Animals", "politics": "Politics",
    "science": "Science", "shopping": "Shopping", "sports": "Sports",
    "technology": "Technology", "travel_transport": "Travel & Transport",
}

def category_from_path(path, root=RAW_AIOS):
    folder = path.relative_to(root).parts[1]
    slug = re.match(r"trending_US_1d_(.+?)_\d{4}-", folder).group(1)
    return CATEGORY_NAMES[slug]

def hostname(url):
    host = (urlparse(url).hostname or "").lower()
    return host[4:] if host.startswith("www.") else host

def show_table(frame, formats=None):
    output = frame.copy()
    for column, template in (formats or {}).items():
        output[column] = output[column].map(template.format)
    display(HTML(output.to_html(index=False, border=0)))
import hashlib

UGC_HOSTS = {"facebook.com", "instagram.com", "linkedin.com", "pinterest.com", "quora.com", "reddit.com", "threads.com", "threads.net", "tiktok.com", "twitter.com", "x.com", "youtube.com", "m.youtube.com"}

counts = defaultdict(Counter)
for result_path in sorted(RAW_AIOS.rglob("result.json")):
    category = category_from_path(result_path)
    data = json.loads(result_path.read_text(encoding="utf-8"))
    refs = (data.get("aiOverview") or {}).get("references") or []
    for ref in refs:
        if not isinstance(ref, dict) or not ref.get("url"):
            continue
        url = ref["url"]
        host = hostname(url)
        is_ugc = host in UGC_HOSTS
        counts[category]["References"] += 1
        counts[category]["UGC"] += is_ugc
        if not is_ugc:
            page_id = hashlib.md5(url.encode("utf-8")).hexdigest()[:8]
            page_path = result_path.parent / "page_analysis" / f"{page_id}.json"
            if page_path.exists():
                page = json.loads(page_path.read_text(encoding="utf-8"))
                counts[category]["Ads"] += int((page.get("ad_metrics") or {}).get("iframe_ad_count") or 0) > 0

table9 = pd.DataFrame([{"Category":category, **values} for category, values in counts.items()])
table9["UGC share"] = 100 * table9["UGC"] / table9["References"]
table9["Ads"] = 100 * table9["Ads"] / table9["References"]
table9 = table9.sort_values("References", ascending=False)

totals = Counter()
for values in counts.values(): totals.update(values)
assert (totals["References"], totals["UGC"], totals["Ads"]) == (61_212, 8_688, 30_994)
expected_refs = {"Sports":20_595, "Entertainment":10_516, "Business & Finance":7_944, "Science":5_212,
                 "Hobbies & Leisure":4_067, "Other":3_733, "Law & Government":2_708, "Politics":1_436,
                 "Climate":941, "Health":839, "Technology":828, "Games":662, "Food & Drink":561,
                 "Jobs & Education":343, "Travel & Transport":290, "Shopping":232, "Autos & Vehicles":186,
                 "Pets & Animals":67, "Beauty & Fashion":52}
assert dict(zip(table9["Category"], table9["References"])) == expected_refs


## Table 9 — References, UGC, and observed ads

In [2]:
display_table = table9[["Category", "References", "UGC", "UGC share", "Ads"]].copy()
display_table.loc[len(display_table)] = ["All", totals["References"], totals["UGC"], 100*totals["UGC"]/totals["References"], 100*totals["Ads"]/totals["References"]]
show_table(display_table, {"References":"{:,}", "UGC":"{:,}", "UGC share":"{:.2f}%", "Ads":"{:.2f}%"})

Category,References,UGC,UGC share,Ads
Sports,"20,595","2,306",11.20%,60.19%
Entertainment,"10,516","2,039",19.39%,55.50%
Business & Finance,"7,944","1,062",13.37%,39.97%
Science,"5,212",798,15.31%,41.85%
Hobbies & Leisure,"4,067",475,11.68%,63.14%
Other,"3,733",604,16.18%,40.05%
Law & Government,"2,708",394,14.55%,29.47%
Politics,"1,436",200,13.93%,36.56%
Climate,941,87,9.25%,48.46%
Health,839,90,10.73%,27.77%


## Sponsored search ads

In [3]:
SPONSORED = ROOT / "replication" / "supporting" / "sponsored_ads"
near = json.loads((SPONSORED / "near_after_aio.json").read_text(encoding="utf-8"))
above = json.loads((SPONSORED / "above_aio.json").read_text(encoding="utf-8"))
inside = json.loads((SPONSORED / "inside_aio.json").read_text(encoding="utf-8"))
assert (len(near), len(above), len(inside)) == (164, 39, 0)

def sponsored_category(item):
    match = re.search(r"trending_US_1d_([a-z_]+)_2026-", item["path"])
    return CATEGORY_NAMES[match.group(1)]

sponsored_by_category = Counter(sponsored_category(item) for item in near)
assert [sponsored_by_category[x] for x in ["Sports", "Business & Finance", "Entertainment", "Politics", "Law & Government"]] == [42, 34, 16, 13, 11]

summary = pd.DataFrame([
    ["AIO SERPs in near-after index", len(near), 100*len(near)/7_583],
    ["Sponsored ad above AIO", len(above), 100*len(above)/7_583],
    ["Sponsored ad inside AIO", len(inside), 0],
], columns=["Measure", "Count", "Share of AIO SERPs"])
show_table(summary, {"Count":"{:,}", "Share of AIO SERPs":"{:.2f}%"})

paper_categories = ["Sports", "Business & Finance", "Entertainment", "Politics", "Law & Government"]
top_categories = pd.DataFrame(
    [(category, sponsored_by_category[category]) for category in paper_categories],
    columns=["Category", "AIO SERPs in near-after index"],
)
show_table(top_categories)

Measure,Count,Share of AIO SERPs
AIO SERPs in near-after index,164,2.16%
Sponsored ad above AIO,39,0.51%
Sponsored ad inside AIO,0,0.00%


Category,AIO SERPs in near-after index
Sports,42
Business & Finance,34
Entertainment,16
Politics,13
Law & Government,11
